In [ ]:
from google.colab import files

uploaded = files.upload()

Saving flight_delay_processed (1) (1).parquet to flight_delay_processed (1) (1).parquet


In [ ]:
import pandas as pd

FILE_PATH = "/content/flight_delay_processed (1) (1).parquet"

df = pd.read_parquet(FILE_PATH)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

Dataset loaded successfully!
Dataset shape: (5713837, 16)


In [ ]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop(columns=["is_delayed"])
y = df["is_delayed"]

# Exact train/test split used by Member 2
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Full training set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nTraining delayed proportion:", y_train.mean())
print("Testing delayed proportion:", y_test.mean())

Full training set: (4571069, 15)
Test set: (1142768, 15)

Training delayed proportion: 0.17912527682255508
Testing delayed proportion: 0.17912559679655013


In [ ]:
# Exact 500,000-row stratified training sample used for final baseline comparison

X_train_common, _, y_train_common, _ = train_test_split(
    X_train,
    y_train,
    train_size=500_000,
    stratify=y_train,
    random_state=42
)

print("Common training set:", X_train_common.shape)
print("Common target set:", y_train_common.shape)

print("\nCommon training distribution:")
print(y_train_common.value_counts(normalize=True))

print("\nTest distribution:")
print(y_test.value_counts(normalize=True))

Common training set: (500000, 15)
Common target set: (500000,)

Common training distribution:
is_delayed
0    0.820874
1    0.179126
Name: proportion, dtype: float64

Test distribution:
is_delayed
0    0.820874
1    0.179126
Name: proportion, dtype: float64


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Exact feature groups used by Member 2

NUMERIC_FEATURES = [
    "MONTH",
    "DAY",
    "DAY_OF_WEEK",
    "DISTANCE",
    "DEP_HOUR",
    "DEP_MINUTE",
    "ARR_HOUR",
    "ARR_MINUTE",
    "IS_WEEKEND"
]

CATEGORICAL_FEATURES = [
    "AIRLINE",
    "ORIGIN_AIRPORT",
    "DESTINATION_AIRPORT",
    "ROUTE"
]

# Numeric preprocessing
numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combined preprocessing
preprocessor = ColumnTransformer([
    ("numeric", numeric_preprocessor, NUMERIC_FEATURES),
    ("categorical", categorical_preprocessor, CATEGORICAL_FEATURES)
])

print("Preprocessing pipeline created successfully!")
print("\nNumeric features:", len(NUMERIC_FEATURES))
print("Categorical features:", len(CATEGORICAL_FEATURES))

Preprocessing pipeline created successfully!

Numeric features: 9
Categorical features: 4


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

# Use a smaller representative sample for Random Forest hyperparameter search
# This is taken ONLY from the already-created 500,000-row training sample.

X_rf_tune, _, y_rf_tune, _ = train_test_split(
    X_train_common,
    y_train_common,
    train_size=100_000,
    stratify=y_train_common,
    random_state=42
)

print("Random Forest tuning sample:", X_rf_tune.shape)
print("Random Forest tuning target:", y_rf_tune.shape)

# Random Forest pipeline
forest_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

# Hyperparameter search space
forest_params = {
    "model__n_estimators": [150, 250, 400],
    "model__max_depth": [10, 20, 30, None],
    "model__min_samples_leaf": [1, 2, 5],
    "model__max_features": ["sqrt", "log2"]
}

# Randomized hyperparameter search
forest_search = RandomizedSearchCV(
    estimator=forest_pipeline,
    param_distributions=forest_params,
    n_iter=12,
    scoring="f1",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("Starting Random Forest hyperparameter tuning...")
forest_search.fit(X_rf_tune, y_rf_tune)

print("\nRandom Forest tuning completed!")
print("Best parameters:")
print(forest_search.best_params_)

print("\nBest cross-validation F1-score:")
print(forest_search.best_score_)

Random Forest tuning sample: (100000, 15)
Random Forest tuning target: (100000,)
Starting Random Forest hyperparameter tuning...
Fitting 3 folds for each of 12 candidates, totalling 36 fits

Random Forest tuning completed!
Best parameters:
{'model__n_estimators': 400, 'model__min_samples_leaf': 5, 'model__max_features': 'sqrt', 'model__max_depth': None}

Best cross-validation F1-score:
0.36243056647829014


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV

# Logistic Regression pipeline
logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1500,
        class_weight="balanced",
        n_jobs=-1
    ))
])

# Hyperparameter search space
logistic_params = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__solver": ["liblinear", "lbfgs"]
}

# Randomized hyperparameter search
logistic_search = RandomizedSearchCV(
    estimator=logistic_pipeline,
    param_distributions=logistic_params,
    n_iter=8,
    scoring="f1",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("Starting Logistic Regression hyperparameter tuning...")

logistic_search.fit(
    X_train_common,
    y_train_common
)

print("\nLogistic Regression tuning completed!")

print("Best parameters:")
print(logistic_search.best_params_)

print("\nBest cross-validation F1-score:")
print(logistic_search.best_score_)

Starting Logistic Regression hyperparameter tuning...
Fitting 3 folds for each of 8 candidates, totalling 24 fits


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 2.
  warnings.warn(



Logistic Regression tuning completed!
Best parameters:
{'model__solver': 'liblinear', 'model__C': 0.01}

Best cross-validation F1-score:
0.3553051522130534


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

# Store the two tuned models
tuned_candidates = {
    "Tuned Logistic Regression": logistic_search.best_estimator_,
    "Tuned Random Forest": forest_search.best_estimator_
}

tuned_results = []
tuned_predictions = {}

# Evaluate both tuned models on the untouched test set
for model_name, model in tuned_candidates.items():

    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(
        y_test, predictions, zero_division=0
    )
    recall = recall_score(
        y_test, predictions, zero_division=0
    )
    f1 = f1_score(
        y_test, predictions, zero_division=0
    )
    roc_auc = roc_auc_score(
        y_test, probabilities
    )

    tuned_results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC_AUC": roc_auc
    })

    tuned_predictions[model_name] = {
        "predictions": predictions,
        "probabilities": probabilities
    }

    print(f"Accuracy : {accuracy:.6f}")
    print(f"Precision: {precision:.6f}")
    print(f"Recall   : {recall:.6f}")
    print(f"F1-score : {f1:.6f}")
    print(f"ROC-AUC  : {roc_auc:.6f}")

    print("\nClassification Report:")
    print(
        classification_report(
            y_test,
            predictions,
            zero_division=0
        )
    )

# Create final tuned-model comparison table
final_results_df = pd.DataFrame(tuned_results)

final_results_df = final_results_df.sort_values(
    "F1",
    ascending=False
).reset_index(drop=True)

print("\nFINAL TUNED MODEL COMPARISON")
display(final_results_df)


Tuned Logistic Regression
Accuracy : 0.603701
Precision: 0.251529
Recall   : 0.613667
F1-score : 0.356810
ROC-AUC  : 0.646123

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.60      0.71    938069
           1       0.25      0.61      0.36    204699

    accuracy                           0.60   1142768
   macro avg       0.56      0.61      0.54   1142768
weighted avg       0.77      0.60      0.65   1142768


Tuned Random Forest
Accuracy : 0.642560
Precision: 0.266427
Recall   : 0.567746
F1-score : 0.362665
ROC-AUC  : 0.658388

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.66      0.75    938069
           1       0.27      0.57      0.36    204699

    accuracy                           0.64   1142768
   macro avg       0.57      0.61      0.56   1142768
weighted avg       0.77      0.64      0.68   1142768


FINAL TUNED MODEL COMPARISON


,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Tuned Random Forest,0.642560,0.266427,0.567746,0.362665,0.658388
1,Tuned Logistic Regression,0.603701,0.251529,0.613667,0.356810,0.646123


In [ ]:
# Select the final model based on the highest test-set F1-score

best_final_name = final_results_df.iloc[0]["Model"]

final_model = tuned_candidates[best_final_name]

print("Selected final model:", best_final_name)
print("Final test-set F1-score:", final_results_df.iloc[0]["F1"])

Selected final model: Tuned Random Forest
Final test-set F1-score: 0.36266529360825706


In [ ]:
import joblib
import os

# Path for the final model
FINAL_MODEL_PATH = "/content/final_flight_delay_pipeline.joblib"

# Save the complete pipeline
joblib.dump(final_model, FINAL_MODEL_PATH)

print("Final pipeline saved successfully!")
print("Saved at:", FINAL_MODEL_PATH)

# Check file size
file_size_mb = os.path.getsize(FINAL_MODEL_PATH) / (1024 * 1024)
print(f"File size: {file_size_mb:.2f} MB")

Final pipeline saved successfully!
Saved at: /content/final_flight_delay_pipeline.joblib
File size: 49.78 MB


In [ ]:
# Reload the saved pipeline

reloaded_model = joblib.load(FINAL_MODEL_PATH)

print("Saved pipeline reloaded successfully!")
print("Reloaded object type:", type(reloaded_model))

Saved pipeline reloaded successfully!
Reloaded object type: <class 'sklearn.pipeline.Pipeline'>


In [ ]:
# Verify that original and reloaded models produce identical predictions

original_predictions = final_model.predict(X_test.head(100))
reloaded_predictions = reloaded_model.predict(X_test.head(100))

print("Original predictions:")
print(original_predictions)

print("\nReloaded predictions:")
print(reloaded_predictions)

# Verify exact equality
assert np.array_equal(
    original_predictions,
    reloaded_predictions
)

print("\nReload verification passed!")
print("Original and reloaded predictions are identical.")

Original predictions:
[0 0 0 0 1 1 1 1 1 1 0 1 1 0 0 0 1 0 0 0 0 0 1 0 1 1 0 0 0 1 0 0 1 0 0 0 0
 1 0 0 0 1 1 1 1 0 0 1 0 1 0 0 0 0 0 0 1 1 0 1 1 0 0 0 0 1 0 1 0 1 0 0 1 0
 0 0 0 0 0 0 1 0 0 0 0 1 1 0 1 1 0 0 0 0 1 1 0 0 0 1]

Reloaded predictions:
[0 0 0 0 1 1 1 1 1 1 0 1 1 0 0 0 1 0 0 0 0 0 1 0 1 1 0 0 0 1 0 0 1 0 0 0 0
 1 0 0 0 1 1 1 1 0 0 1 0 1 0 0 0 0 0 0 1 1 0 1 1 0 0 0 0 1 0 1 0 1 0 0 1 0
 0 0 0 0 0 0 1 0 0 0 0 1 1 0 1 1 0 0 0 0 1 1 0 0 0 1]

Reload verification passed!
Original and reloaded predictions are identical.


In [ ]:
# Create a completely new/unseen flight record

new_flight = pd.DataFrame({
    "MONTH": [7],
    "DAY": [15],
    "DAY_OF_WEEK": [3],
    "DISTANCE": [760],
    "DEP_HOUR": [17],
    "DEP_MINUTE": [30],
    "ARR_HOUR": [20],
    "ARR_MINUTE": [15],
    "IS_WEEKEND": [0],
    "AIRLINE": ["AA"],
    "ORIGIN_AIRPORT": ["JFK"],
    "DESTINATION_AIRPORT": ["LAX"],
    "ROUTE": ["JFK_TO_LAX"]
})

print("New unseen flight:")
display(new_flight)

New unseen flight:


,MONTH,DAY,DAY_OF_WEEK,DISTANCE,DEP_HOUR,DEP_MINUTE,ARR_HOUR,ARR_MINUTE,IS_WEEKEND,AIRLINE,ORIGIN_AIRPORT,DESTINATION_AIRPORT,ROUTE
0,7,15,3,760,17,30,20,15,0,AA,JFK,LAX,JFK_TO_LAX


In [ ]:
# Make prediction using the reloaded final pipeline

prediction = reloaded_model.predict(new_flight)[0]

# Get probability of delay
delay_probability = reloaded_model.predict_proba(new_flight)[0, 1]

print("Prediction result:")
print("------------------")

if prediction == 1:
    print("The model predicts that the flight MAY BE DELAYED by more than 15 minutes.")
else:
    print("The model predicts that the flight MAY NOT BE DELAYED by more than 15 minutes.")

print(f"Estimated probability of delay: {delay_probability:.2%}")

Prediction result:
------------------
The model predicts that the flight MAY BE DELAYED by more than 15 minutes.
Estimated probability of delay: 59.53%


In [ ]:
try:
    import xgboost as xgb
    print("XGBoost is installed.")
    print("Version:", xgb.__version__)
except ImportError:
    print("XGBoost is not installed.")

XGBoost is installed.
Version: 3.4.1


In [ ]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV

# XGBoost model
xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

# Complete pipeline
xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", xgb_model)
])

print("XGBoost pipeline created successfully.")

NameError: name 'preprocessor' is not defined

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from xgboost import XGBClassifier

# Same features used in our previous models
numeric_features = [
    "MONTH",
    "DAY",
    "DAY_OF_WEEK",
    "DISTANCE",
    "DEP_HOUR",
    "DEP_MINUTE",
    "ARR_HOUR",
    "ARR_MINUTE",
    "IS_WEEKEND"
]

categorical_features = [
    "AIRLINE",
    "ORIGIN_AIRPORT",
    "DESTINATION_AIRPORT",
    "ROUTE"
]

# Numeric preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combined preprocessing
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# XGBoost model
xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

# Complete pipeline
xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", xgb_model)
])

print("XGBoost pipeline created successfully.")
print("Preprocessing + XGBoost are ready.")

XGBoost pipeline created successfully.
Preprocessing + XGBoost are ready.


In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# XGBoost hyperparameter search space
xgb_param_grid = {
    "model__n_estimators": [200, 300, 400],
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0],
    "model__min_child_weight": [1, 5, 10]
}

# Randomized search
xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=xgb_param_grid,
    n_iter=8,
    scoring="f1",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=2
)

print("Starting XGBoost hyperparameter tuning...")
print("This may take some time because the dataset is large.")

xgb_search.fit(X_rf_tune, y_rf_tune)

print("\nXGBoost tuning completed!")
print("Best parameters:")
print(xgb_search.best_params_)
print("\nBest CV F1:", xgb_search.best_score_)

Starting XGBoost hyperparameter tuning...
This may take some time because the dataset is large.


NameError: name 'X_rf_tune' is not defined

In [ ]:
from sklearn.model_selection import train_test_split

# Recreate the same 100,000-row stratified tuning sample
X_xgb_tune, _, y_xgb_tune, _ = train_test_split(
    X_common_train,
    y_common_train,
    train_size=100000,
    stratify=y_common_train,
    random_state=42
)

print("XGBoost tuning sample created successfully.")
print("Shape:", X_xgb_tune.shape)
print("Target shape:", y_xgb_tune.shape)
print("\nClass distribution:")
print(y_xgb_tune.value_counts(normalize=True))

NameError: name 'X_common_train' is not defined

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the processed dataset
DATA_PATH = "/content/flight_delay_processed (1) (1).parquet"

df = pd.read_parquet(DATA_PATH)

print("Dataset loaded:", df.shape)

# Separate features and target
X = df.drop(columns=["is_delayed"])
y = df["is_delayed"]

# Exact same 80/20 stratified split used earlier
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

# Exact same 500,000-row common training sample
X_common_train, _, y_common_train, _ = train_test_split(
    X_train,
    y_train,
    train_size=500000,
    stratify=y_train,
    random_state=42
)

print("Common training sample:", X_common_train.shape)

# 100,000-row XGBoost tuning sample
X_xgb_tune, _, y_xgb_tune, _ = train_test_split(
    X_common_train,
    y_common_train,
    train_size=100000,
    stratify=y_common_train,
    random_state=42
)

print("XGBoost tuning sample:", X_xgb_tune.shape)
print("Target distribution:")
print(y_xgb_tune.value_counts(normalize=True))

FileNotFoundError: [Errno 2] No such file or directory: '/content/flight_delay_processed (1) (1).parquet'

In [ ]:
import os

print("Files currently in /content:")
for file in os.listdir("/content"):
    print(file)

Files currently in /content:
.config
sample_data


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving flight_delay_processed (1) (1).parquet to flight_delay_processed (1) (1).parquet


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load processed dataset
DATA_PATH = "/content/flight_delay_processed (1) (1).parquet"

df = pd.read_parquet(DATA_PATH)

print("Dataset shape:", df.shape)

# Separate features and target
X = df.drop(columns=["is_delayed"])
y = df["is_delayed"]

# Exact same 80/20 stratified split used earlier
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

# Exact same 500,000-row common training sample
X_common_train, _, y_common_train, _ = train_test_split(
    X_train,
    y_train,
    train_size=500000,
    stratify=y_train,
    random_state=42
)

print("Common training sample:", X_common_train.shape)

# 100,000-row XGBoost tuning sample
X_xgb_tune, _, y_xgb_tune, _ = train_test_split(
    X_common_train,
    y_common_train,
    train_size=100000,
    stratify=y_common_train,
    random_state=42
)

print("XGBoost tuning sample:", X_xgb_tune.shape)
print("\nTarget distribution:")
print(y_xgb_tune.value_counts(normalize=True))

Dataset shape: (5713837, 16)
Training set: (4571069, 15)
Test set: (1142768, 15)
Common training sample: (500000, 15)
XGBoost tuning sample: (100000, 15)

Target distribution:
is_delayed
0    0.82087
1    0.17913
Name: proportion, dtype: float64


In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# XGBoost hyperparameter search space
xgb_param_grid = {
    "model__n_estimators": [200, 300, 400],
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0],
    "model__min_child_weight": [1, 5, 10]
}

# Randomized hyperparameter search
xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=xgb_param_grid,
    n_iter=8,
    scoring="f1",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=2
)

print("Starting XGBoost hyperparameter tuning...")
print("8 parameter combinations × 3-fold CV = 24 model fits.")

xgb_search.fit(X_xgb_tune, y_xgb_tune)

print("\nXGBoost tuning completed!")
print("\nBest parameters:")
print(xgb_search.best_params_)

print("\nBest CV F1:")
print(xgb_search.best_score_)

Starting XGBoost hyperparameter tuning...
8 parameter combinations × 3-fold CV = 24 model fits.
Fitting 3 folds for each of 8 candidates, totalling 24 fits

XGBoost tuning completed!

Best parameters:
{'model__subsample': 1.0, 'model__n_estimators': 400, 'model__min_child_weight': 5, 'model__max_depth': 5, 'model__learning_rate': 0.1, 'model__colsample_bytree': 0.8}

Best CV F1:
0.03974375238131186


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Best tuned XGBoost model
best_xgb = xgb_search.best_estimator_

print("Evaluating XGBoost on the untouched test set...")
print("This may take some time.")

# Predictions
xgb_predictions = best_xgb.predict(X_test)
xgb_probabilities = best_xgb.predict_proba(X_test)[:, 1]

# Metrics
xgb_accuracy = accuracy_score(y_test, xgb_predictions)
xgb_precision = precision_score(y_test, xgb_predictions)
xgb_recall = recall_score(y_test, xgb_predictions)
xgb_f1 = f1_score(y_test, xgb_predictions)
xgb_roc_auc = roc_auc_score(y_test, xgb_probabilities)

print("\n========== XGBOOST TEST RESULTS ==========")
print(f"Accuracy : {xgb_accuracy:.6f}")
print(f"Precision: {xgb_precision:.6f}")
print(f"Recall   : {xgb_recall:.6f}")
print(f"F1 Score : {xgb_f1:.6f}")
print(f"ROC-AUC  : {xgb_roc_auc:.6f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, xgb_predictions))

Evaluating XGBoost on the untouched test set...
This may take some time.

========== XGBOOST TEST RESULTS ==========
Accuracy : 0.821670
Precision: 0.585192
Recall   : 0.015252
F1 Score : 0.029729
ROC-AUC  : 0.684392

Confusion Matrix:
[[935856   2213]
 [201577   3122]]


In [ ]:
from sklearn.model_selection import train_test_split

# Split the 500,000-row common training sample
# into training and validation sets.
X_rf_train, X_rf_val, y_rf_train, y_rf_val = train_test_split(
    X_common_train,
    y_common_train,
    test_size=0.20,
    stratify=y_common_train,
    random_state=42
)

print("Random Forest training set:", X_rf_train.shape)
print("Random Forest validation set:", X_rf_val.shape)

print("\nValidation class distribution:")
print(y_rf_val.value_counts(normalize=True))

Random Forest training set: (400000, 15)
Random Forest validation set: (100000, 15)

Validation class distribution:
is_delayed
0    0.82087
1    0.17913
Name: proportion, dtype: float64


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

# Tuned Random Forest
rf_validation_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=5,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

print("Training tuned Random Forest on 400,000 training rows...")
print("Please wait; this may take some time.")

rf_validation_model.fit(X_rf_train, y_rf_train)

print("\nTuned Random Forest training completed successfully!")

Training tuned Random Forest on 400,000 training rows...
Please wait; this may take some time.

Tuned Random Forest training completed successfully!


In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

# Get probability of "Delayed" class
val_probabilities = rf_validation_model.predict_proba(X_rf_val)[:, 1]

# Test different classification thresholds
thresholds = np.arange(0.20, 0.71, 0.05)

results = []

for threshold in thresholds:
    val_predictions = (val_probabilities >= threshold).astype(int)

    precision = precision_score(y_rf_val, val_predictions, zero_division=0)
    recall = recall_score(y_rf_val, val_predictions, zero_division=0)
    f1 = f1_score(y_rf_val, val_predictions, zero_division=0)

    results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })

threshold_results = pd.DataFrame(results)

print(threshold_results.to_string(index=False))

# Find threshold with highest validation F1
best_threshold_row = threshold_results.loc[
    threshold_results["F1"].idxmax()
]

print("\n========== BEST VALIDATION THRESHOLD ==========")
print(f"Threshold : {best_threshold_row['Threshold']:.2f}")
print(f"Precision : {best_threshold_row['Precision']:.6f}")
print(f"Recall    : {best_threshold_row['Recall']:.6f}")
print(f"F1 Score  : {best_threshold_row['F1']:.6f}")

 Threshold  Precision   Recall       F1
      0.20   0.179440 0.999777 0.304270
      0.25   0.180973 0.997823 0.306379
      0.30   0.187718 0.984592 0.315318
      0.35   0.200239 0.944230 0.330410
      0.40   0.218989 0.867973 0.349739
      0.45   0.243638 0.739686 0.366544
      0.50   0.273750 0.573159 0.370529
      0.55   0.310699 0.370122 0.337817
      0.60   0.358697 0.162898 0.224048
      0.65   0.460894 0.036845 0.068235
      0.70   0.546667 0.002289 0.004559

========== BEST VALIDATION THRESHOLD ==========
Threshold : 0.50
Precision : 0.273750
Recall    : 0.573159
F1 Score  : 0.370529


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

# Predict on the completely untouched test set
test_predictions_val_model = rf_validation_model.predict(X_test)
test_probabilities_val_model = rf_validation_model.predict_proba(X_test)[:, 1]

# Calculate metrics
final_accuracy = accuracy_score(y_test, test_predictions_val_model)
final_precision = precision_score(y_test, test_predictions_val_model)
final_recall = recall_score(y_test, test_predictions_val_model)
final_f1 = f1_score(y_test, test_predictions_val_model)
final_roc_auc = roc_auc_score(y_test, test_probabilities_val_model)

print("========== RANDOM FOREST FINAL TEST RESULTS ==========")
print(f"Accuracy : {final_accuracy:.6f}")
print(f"Precision: {final_precision:.6f}")
print(f"Recall   : {final_recall:.6f}")
print(f"F1 Score : {final_f1:.6f}")
print(f"ROC-AUC  : {final_roc_auc:.6f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_predictions_val_model))

========== RANDOM FOREST FINAL TEST RESULTS ==========
Accuracy : 0.653783
Precision: 0.276732
Recall   : 0.578098
F1 Score : 0.374293
ROC-AUC  : 0.674261

Confusion Matrix:
[[628786 309283]
 [ 86363 118336]]


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

# Final tuned Random Forest
final_rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=5,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

print("Training FINAL Random Forest on 500,000 rows...")
print("Please wait; this may take some time.")

final_rf_pipeline.fit(X_common_train, y_common_train)

print("\nFINAL Random Forest training completed successfully!")

NameError: name 'preprocessor' is not defined

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

# Final tuned Random Forest
final_rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=5,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

print("Training FINAL Random Forest on 500,000 rows...")
print("Please wait; this may take some time.")

final_rf_pipeline.fit(X_common_train, y_common_train)

print("\nFINAL Random Forest training completed successfully!")

NameError: name 'preprocessor' is not defined

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Same features used throughout our project
numeric_features = [
    "MONTH",
    "DAY",
    "DAY_OF_WEEK",
    "DISTANCE",
    "DEP_HOUR",
    "DEP_MINUTE",
    "ARR_HOUR",
    "ARR_MINUTE",
    "IS_WEEKEND"
]

categorical_features = [
    "AIRLINE",
    "ORIGIN_AIRPORT",
    "DESTINATION_AIRPORT",
    "ROUTE"
]

# Numeric preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combined preprocessing
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Preprocessor recreated successfully.")

Preprocessor recreated successfully.


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

# Final tuned Random Forest
final_rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=5,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

print("Training FINAL Random Forest on 500,000 rows...")
print("Please wait; this may take a few minutes.")

final_rf_pipeline.fit(X_common_train, y_common_train)

print("\nFINAL Random Forest training completed successfully!")

Training FINAL Random Forest on 500,000 rows...
Please wait; this may take a few minutes.


NameError: name 'X_common_train' is not defined

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load processed dataset
DATA_PATH = "/content/flight_delay_processed (1) (1).parquet"

df = pd.read_parquet(DATA_PATH)

# Separate features and target
X = df.drop(columns=["is_delayed"])
y = df["is_delayed"]

# Exact project split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

# Exact 500,000-row common training sample
X_common_train, _, y_common_train, _ = train_test_split(
    X_train,
    y_train,
    train_size=500000,
    stratify=y_train,
    random_state=42
)

print("Dataset:", df.shape)
print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("Final training sample:", X_common_train.shape)
print("Target distribution:")
print(y_common_train.value_counts(normalize=True))

FileNotFoundError: [Errno 2] No such file or directory: '/content/flight_delay_processed (1) (1).parquet'

In [6]:
from google.colab import files

uploaded = files.upload()

print("\nUploaded files:")
for filename in uploaded.keys():
    print(filename)

Saving flight_delay_processed (1) (1).parquet to flight_delay_processed (1) (1).parquet

Uploaded files:
flight_delay_processed (1) (1).parquet


In [7]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# Automatically find the uploaded parquet file
parquet_files = [
    f for f in os.listdir("/content")
    if f.endswith(".parquet")
]

print("Parquet files found:", parquet_files)

if not parquet_files:
    raise FileNotFoundError("No .parquet file found in /content.")

DATA_PATH = "/content/" + parquet_files[0]

# Load dataset
df = pd.read_parquet(DATA_PATH)

print("\nDataset shape:", df.shape)

# Separate features and target
X = df.drop(columns=["is_delayed"])
y = df["is_delayed"]

# Exact same 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

# Exact same 500,000-row training sample
X_common_train, _, y_common_train, _ = train_test_split(
    X_train,
    y_train,
    train_size=500000,
    stratify=y_train,
    random_state=42
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("Final training sample:", X_common_train.shape)

print("\nTarget distribution:")
print(y_common_train.value_counts(normalize=True))

Parquet files found: ['flight_delay_processed (1) (1).parquet']

Dataset shape: (5713837, 16)
Training set: (4571069, 15)
Test set: (1142768, 15)
Final training sample: (500000, 15)

Target distribution:
is_delayed
0    0.820874
1    0.179126
Name: proportion, dtype: float64


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

# Recreate the preprocessor in case the runtime lost it
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features = [
    "MONTH",
    "DAY",
    "DAY_OF_WEEK",
    "DISTANCE",
    "DEP_HOUR",
    "DEP_MINUTE",
    "ARR_HOUR",
    "ARR_MINUTE",
    "IS_WEEKEND"
]

categorical_features = [
    "AIRLINE",
    "ORIGIN_AIRPORT",
    "DESTINATION_AIRPORT",
    "ROUTE"
]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Final tuned Random Forest
final_rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=5,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

print("Training FINAL Random Forest on 500,000 rows...")
print("Please wait; this may take a few minutes.")

final_rf_pipeline.fit(X_common_train, y_common_train)

print("\nFINAL Random Forest training completed successfully!")

Training FINAL Random Forest on 500,000 rows...
Please wait; this may take a few minutes.

FINAL Random Forest training completed successfully!


In [9]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

print("Evaluating FINAL Random Forest on untouched test set...")
print("Please wait; this may take some time.")

# Predictions
final_predictions = final_rf_pipeline.predict(X_test)
final_probabilities = final_rf_pipeline.predict_proba(X_test)[:, 1]

# Calculate final metrics
final_accuracy = accuracy_score(y_test, final_predictions)
final_precision = precision_score(y_test, final_predictions)
final_recall = recall_score(y_test, final_predictions)
final_f1 = f1_score(y_test, final_predictions)
final_roc_auc = roc_auc_score(y_test, final_probabilities)

print("\n========== FINAL RANDOM FOREST RESULTS ==========")
print(f"Accuracy : {final_accuracy:.6f}")
print(f"Precision: {final_precision:.6f}")
print(f"Recall   : {final_recall:.6f}")
print(f"F1 Score : {final_f1:.6f}")
print(f"ROC-AUC  : {final_roc_auc:.6f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, final_predictions))

Evaluating FINAL Random Forest on untouched test set...
Please wait; this may take some time.

========== FINAL RANDOM FOREST RESULTS ==========
Accuracy : 0.657273
Precision: 0.278599
Recall   : 0.574649
F1 Score : 0.375264
ROC-AUC  : 0.676230

Confusion Matrix:
[[633480 304589]
 [ 87069 117630]]


In [10]:
import joblib
import os

FINAL_MODEL_PATH = "/content/final_flight_delay_pipeline.joblib"

# Save the complete pipeline
joblib.dump(final_rf_pipeline, FINAL_MODEL_PATH)

print("Final pipeline saved successfully!")
print("Path:", FINAL_MODEL_PATH)
print("File size:", round(os.path.getsize(FINAL_MODEL_PATH) / (1024 * 1024), 2), "MB")

Final pipeline saved successfully!
Path: /content/final_flight_delay_pipeline.joblib
File size: 414.43 MB


In [11]:
import joblib
import numpy as np

# Reload the saved pipeline
reloaded_model = joblib.load(
    "/content/final_flight_delay_pipeline.joblib"
)

print("Saved pipeline reloaded successfully!")

# Compare predictions from original and reloaded models
original_predictions = final_rf_pipeline.predict(X_test.iloc[:100])
reloaded_predictions = reloaded_model.predict(X_test.iloc[:100])

# Verify they are exactly identical
if np.array_equal(original_predictions, reloaded_predictions):
    print("Reload verification PASSED!")
    print("Original and reloaded predictions are identical.")
else:
    print("Reload verification FAILED!")

Saved pipeline reloaded successfully!
Reload verification PASSED!
Original and reloaded predictions are identical.


In [12]:
import pandas as pd

# Completely new/unseen flight
new_flight = pd.DataFrame({
    "MONTH": [7],
    "DAY": [15],
    "DAY_OF_WEEK": [3],
    "DISTANCE": [760],
    "DEP_HOUR": [17],
    "DEP_MINUTE": [30],
    "ARR_HOUR": [20],
    "ARR_MINUTE": [15],
    "IS_WEEKEND": [0],
    "AIRLINE": ["AA"],
    "ORIGIN_AIRPORT": ["JFK"],
    "DESTINATION_AIRPORT": ["LAX"],
    "ROUTE": ["JFK_TO_LAX"]
})

# Prediction using the RELOADED saved pipeline
prediction = reloaded_model.predict(new_flight)[0]
delay_probability = reloaded_model.predict_proba(new_flight)[0, 1]

print("========== UNSEEN FLIGHT PREDICTION ==========")

print("Flight details:")
print(new_flight.to_string(index=False))

print("\nPrediction:")
if prediction == 1:
    print("Flight is predicted to be DELAYED")
else:
    print("Flight is predicted to be NOT DELAYED")

print(f"\nProbability of delay: {delay_probability:.4%}")

========== UNSEEN FLIGHT PREDICTION ==========
Flight details:
 MONTH  DAY  DAY_OF_WEEK  DISTANCE  DEP_HOUR  DEP_MINUTE  ARR_HOUR  ARR_MINUTE  IS_WEEKEND AIRLINE ORIGIN_AIRPORT DESTINATION_AIRPORT      ROUTE
     7   15            3       760        17          30        20          15           0      AA            JFK                 LAX JFK_TO_LAX

Prediction:
Flight is predicted to be DELAYED

Probability of delay: 59.9671%
